In [ ]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt

In [ ]:
!cat /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt

In [ ]:
# ============================================================================
# CELL 1 — SETUP & BLACKWELL OPTIMIZATIONS (ColPali v1.3)
# ============================================================================
import os, gc, io, json, glob, time, shutil, pickle, traceback
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# ----------------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------------
VIDORE_ROOT = "/kaggle/input/datasets/namthi/vidore-v3"
MODEL_ROOT  = "/kaggle/input/datasets/guiniever/colpali-v1-3-offline/colpali_assets"
ADAPTER_DIR = os.path.join(MODEL_ROOT, "colpali-v1.3")
BASE_DIR    = os.path.join(MODEL_ROOT, "vidore_colpaligemma-3b-pt-448-base")

OUTPUT_DIR     = "/kaggle/working/vidore_encoded"
CHECKPOINT_DIR = "/kaggle/working/vidore_encoded/_checkpoints"
PATCHED_ADAPTER_DIR = "/kaggle/working/_colpali_adapter_patched"

os.makedirs(OUTPUT_DIR,     exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ----------------------------------------------------------------------------
# ENCODING CONFIG
# ----------------------------------------------------------------------------
BATCH_SIZE          = 16
SAVE_EVERY_N_PAGES  = 200
EMBED_STORAGE_DTYPE = torch.bfloat16
RUNTIME_DTYPE       = torch.bfloat16

# PaliGemma dùng fixed 448×448 resolution → ~1024 visual tokens
# Không có max_num_visual_tokens parameter như ColQwen2

# ----------------------------------------------------------------------------
# BLACKWELL OPTIMIZATIONS
# ----------------------------------------------------------------------------
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# ----------------------------------------------------------------------------
# SANITY CHECKS
# ----------------------------------------------------------------------------
assert os.path.isdir(VIDORE_ROOT), f"Không tìm thấy VIDORE_ROOT: {VIDORE_ROOT}"
assert os.path.isdir(ADAPTER_DIR), f"Không tìm thấy ADAPTER_DIR: {ADAPTER_DIR}"
assert os.path.isdir(BASE_DIR),    f"Không tìm thấy BASE_DIR: {BASE_DIR}"
assert torch.cuda.is_available(),  "CUDA không khả dụng"

device = "cuda:0"
print(f">>> GPU             : {torch.cuda.get_device_name(0)}")
print(f">>> VRAM            : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f">>> Runtime dtype   : {RUNTIME_DTYPE}")
print(f">>> BATCH_SIZE      : {BATCH_SIZE}")
print(f">>> Model           : ColPali v1.3 (PaliGemma-3B)")
print(f">>> Output dir      : {OUTPUT_DIR}")

In [ ]:
# ============================================================================
# CELL 2 — LOAD COLPALI v1.3 (offline-safe)
# Patch adapter_config.json base_model_name_or_path → local BASE_DIR
# ============================================================================
print(">>> CELL 2 — Loading ColPali v1.3 (offline)")

from colpali_engine.models import ColPali, ColPaliProcessor

# Step 1: patch adapter dir
if os.path.isdir(PATCHED_ADAPTER_DIR):
    shutil.rmtree(PATCHED_ADAPTER_DIR)
shutil.copytree(ADAPTER_DIR, PATCHED_ADAPTER_DIR)

cfg_path = os.path.join(PATCHED_ADAPTER_DIR, "adapter_config.json")
with open(cfg_path, "r") as f:
    adapter_cfg = json.load(f)
original_base = adapter_cfg.get("base_model_name_or_path")
adapter_cfg["base_model_name_or_path"] = BASE_DIR
with open(cfg_path, "w") as f:
    json.dump(adapter_cfg, f, indent=2)
print(f">>> Patched: {original_base} → {BASE_DIR}")

# Step 2: load processor (không có max_num_visual_tokens — PaliGemma fixed 448px)
processor = ColPaliProcessor.from_pretrained(PATCHED_ADAPTER_DIR)
print(f">>> Processor loaded")

def verify_lora_loaded(model):
    found = {}
    for name, param in model.named_parameters():
        if "custom_text_proj" not in name:
            continue
        n = param.detach().float().norm().item()
        is_critical = ("lora_A" in name and "weight" in name) or \
                      ("lora_B" in name and "weight" in name)
        found[name] = (tuple(param.shape), n, is_critical)

    print("\n>>> custom_text_proj parameters:")
    all_critical_ok = True
    n_critical = 0
    for k, (shape, n, is_critical) in found.items():
        if is_critical:
            n_critical += 1
            ok = n > 1e-4
            flag = "✅" if ok else "❌ ZERO (critical)"
            if not ok:
                all_critical_ok = False
        else:
            flag = "ℹ️  (non-critical)"
        print(f"   {flag}  {k:60s}  shape={shape}  norm={n:.6f}")
    return all_critical_ok and n_critical >= 2

# Step 3: load BASE model trước (không qua adapter)
model = ColPali.from_pretrained(
    BASE_DIR,
    torch_dtype=RUNTIME_DTYPE,
    device_map=device,
    attn_implementation="eager",
    low_cpu_mem_usage=True,
).eval()
print(f">>> Base model loaded on {next(model.parameters()).device}")

# Step 4: apply adapter weights thủ công
from safetensors.torch import load_file as safe_load

sf_files = sorted(glob.glob(os.path.join(PATCHED_ADAPTER_DIR, "adapter_model*.safetensors")))
raw = {}
for f in sf_files:
    raw.update(safe_load(f, device="cpu"))
print(f">>> Loaded {len(raw)} adapter keys from safetensors")

# Build name mapping: adapter key → model param name
# Adapter keys: "base_model.model.{rest}" → model param: "{rest}"
# Ví dụ: "base_model.model.custom_text_proj.lora_A.weight" → "custom_text_proj.lora_A.weight"
#         "base_model.model.model.language_model.model.layers.0..." → "model.language_model.model.layers.0..."
PREFIX = "base_model.model."

model_params = dict(model.named_parameters())
applied = 0
skipped = 0

with torch.no_grad():
    for adapter_key, adapter_val in raw.items():
        # Strip prefix
        if adapter_key.startswith(PREFIX):
            model_key = adapter_key[len(PREFIX):]
        else:
            model_key = adapter_key

        if model_key in model_params:
            param = model_params[model_key]
            if param.shape == adapter_val.shape:
                param.data.copy_(adapter_val.to(dtype=param.dtype, device=param.device))
                applied += 1
            else:
                print(f"  ⚠️ Shape mismatch: {model_key} "
                      f"ckpt={tuple(adapter_val.shape)} vs model={tuple(param.shape)}")
                skipped += 1
        else:
            skipped += 1

print(f">>> Applied: {applied} / {len(raw)} params")
print(f">>> Skipped: {skipped} (expected — nhiều LoRA keys cần PEFT wrapper)")

# Step 5: nếu direct apply không đủ (LoRA keys cần PEFT wrap),
# thử load_adapter với workaround monkey-patch
if applied < len(raw) * 0.5:
    print(f"\n>>> Direct apply chỉ được {applied}/{len(raw)} — thử PEFT load_adapter với monkey-patch...")
    try:
        # Monkey-patch để skip MOE conversion bug
        import transformers.integrations.peft as peft_integration
        _original_convert = peft_integration.convert_peft_config_for_transformers

        def _patched_convert(peft_config, model=None, conversions=None):
            try:
                return _original_convert(peft_config, model=model, conversions=conversions)
            except KeyError:
                return peft_config

        peft_integration.convert_peft_config_for_transformers = _patched_convert

        model.load_adapter(PATCHED_ADAPTER_DIR)
        print(">>> load_adapter thành công (với monkey-patch)")

        # Restore
        peft_integration.convert_peft_config_for_transformers = _original_convert
    except Exception as e:
        print(f">>> load_adapter cũng fail: {e}")
        print(">>> Tiếp tục với direct-applied weights")

# Step 6: verify LoRA
lora_ok = verify_lora_loaded(model)
if not lora_ok:
    print("\n⚠️ LoRA vẫn chưa load — fallback patch custom_text_proj trực tiếp")
    proj_keys = sorted([k for k in raw.keys() if "custom_text_proj" in k])
    key_a = next((k for k in proj_keys if "lora_A" in k), None)
    key_b = next((k for k in proj_keys if "lora_B" in k), None)

    target_a, target_b = None, None
    target_a_name, target_b_name = None, None
    for name, param in model.named_parameters():
        if "custom_text_proj" not in name:
            continue
        if "lora_A" in name and "weight" in name and target_a is None:
            target_a, target_a_name = param, name
        if "lora_B" in name and "weight" in name and target_b is None:
            target_b, target_b_name = param, name

    if target_a is not None and target_b is not None and key_a and key_b:
        with torch.no_grad():
            target_a.data.copy_(raw[key_a].to(dtype=target_a.dtype, device=target_a.device))
            target_b.data.copy_(raw[key_b].to(dtype=target_b.dtype, device=target_b.device))
        print(f">>> Patched: {key_a} → {target_a_name}")
        print(f">>> Patched: {key_b} → {target_b_name}")
        assert verify_lora_loaded(model), "❌ Final patch FAIL"
        print("✅ Fallback patch thành công")
    else:
        print("❌ Không tìm thấy LoRA targets trong model — có thể base model chưa có PEFT wrapper")
        print("   Model sẽ chạy với base weights only (chất lượng sẽ thấp hơn)")
else:
    print("\n✅ Adapter weights loaded OK")

print("\n>>> CELL 2 DONE")

In [ ]:
print(type(model))
for name, child in model.named_children():
    print(f"  {name}: {type(child)}")

In [ ]:
# ============================================================================
# CELL 1 — SHARED UTILITIES FOR PIPELINE B (ViDoRe V3 × ColQwen2)
# - Discover domains, load encoded PKLs, build global doc_matrix (cross-domain)
# - Load queries + qrels, map GT to global indices
# - Core MaxSim, content mask, metrics helpers, latency timer
# ============================================================================
import os
import gc
import io
import glob
import json
import time
import pickle

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
VIDORE_ROOT     = "/kaggle/input/datasets/namthi/vidore-v3"
ENCODED_ROOT = "/kaggle/input/datasets/guiniever/vidore-colpali-encoded"  # ← pkl ColPali
OUTPUT_DIR      = "/kaggle/working/pipelineB_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TOPK_RATIO = 0.5            # 50% cố định cho tất cả method (pruning & pooling)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"

# ----------------------------------------------------------------------------
# DISCOVER DOMAINS — scan VIDORE_ROOT/vidore_v3_*/vidore_v3_*/{corpus,queries,qrels}
# ----------------------------------------------------------------------------
def discover_domains(vidore_root):
    domains = []
    outer_dirs = sorted(glob.glob(os.path.join(vidore_root, "vidore_v3_*")))
    for outer in outer_dirs:
        if not os.path.isdir(outer):
            continue
        domain_name = os.path.basename(outer).replace("vidore_v3_", "")
        candidates = [
            os.path.join(outer, os.path.basename(outer)),  # double-nested (Kaggle)
            outer,                                          # flat (fallback)
        ]
        base = next((c for c in candidates
                     if os.path.isdir(os.path.join(c, "corpus"))), None)
        if base is None:
            continue
        corpus_files  = sorted(glob.glob(os.path.join(base, "corpus",  "*.parquet")))
        queries_files = sorted(glob.glob(os.path.join(base, "queries", "*.parquet")))
        qrels_files   = sorted(glob.glob(os.path.join(base, "qrels",   "*.parquet")))
        if not (corpus_files and queries_files and qrels_files):
            print(f"  ⚠️  {domain_name}: thiếu một trong corpus/queries/qrels — skip")
            continue
        domains.append({
            "name":          domain_name,
            "base":          base,
            "corpus_files":  corpus_files,
            "queries_files": queries_files,
            "qrels_files":   qrels_files,
        })
    return domains


domains = discover_domains(VIDORE_ROOT)
print(f">>> Found {len(domains)} domains:")
for d in domains:
    print(f"  - {d['name']}")

# ----------------------------------------------------------------------------
# LOAD ALL PIPELINE-A PKLS → BUILD GLOBAL INDEX
# Mỗi item trong pkl có schema:
#   {embedding [S, 128] bf16, corpus_id, doc_id, page_number_in_doc, domain}
# ----------------------------------------------------------------------------
def load_all_encoded_pkls(encoded_root, domain_names):
    """
    Trả về:
        global_items: list[dict]
        domain_to_global: dict {(domain, local_corpus_id): global_idx}
    """
    global_items = []
    domain_to_global = {}
    for dname in domain_names:
        pkl_path = os.path.join(encoded_root, f"vidore_{dname}.pkl")
        if not os.path.exists(pkl_path):
            print(f"  ⚠️  PKL không tồn tại: {pkl_path}")
            continue
        with open(pkl_path, 'rb') as f:
            items = pickle.load(f)
        print(f"  {dname:25s}: {len(items)} pages")
        for item in items:
            gidx = len(global_items)
            global_items.append({
                'embedding':          item['embedding'],
                'domain':             dname,
                'corpus_id_local':    int(item['corpus_id']),
                'doc_id':             item['doc_id'],
                'page_number_in_doc': item['page_number_in_doc'],
                'global_idx':         gidx,
            })
            domain_to_global[(dname, int(item['corpus_id']))] = gidx
    return global_items, domain_to_global


print("\n>>> Loading encoded PKLs...")
global_items, domain_to_global = load_all_encoded_pkls(
    ENCODED_ROOT, [d['name'] for d in domains]
)
print(f">>> Total pages in global corpus: {len(global_items)}")

# ----------------------------------------------------------------------------
# BUILD GLOBAL DOC MATRIX — padded [N, max_len, 128]
# ----------------------------------------------------------------------------
def build_global_doc_matrix(global_items, device, dtype=torch.float32):
    """
    Padded matrix + mask. Embeddings đã L2-normalized để dùng trực tiếp trong MaxSim.
    Dùng float32 trên GPU để MaxSim chính xác; storage gốc là bf16 nên chỉ upcast.
    """
    tensors = [it['embedding'] for it in global_items]
    n     = len(tensors)
    max_l = max(t.shape[0] for t in tensors)
    D     = tensors[0].shape[1]
    print(f">>> Building doc_matrix: [{n}, {max_l}, {D}]  "
          f"(~{n*max_l*D*4/1e9:.2f} GB float32)")

    doc_matrix = torch.zeros(n, max_l, D, device=device, dtype=dtype)
    doc_mask   = torch.zeros(n, max_l,    device=device, dtype=torch.bool)

    for i, t in enumerate(tqdm(tensors, desc="Filling doc_matrix", leave=False)):
        L = t.shape[0]
        doc_matrix[i, :L] = F.normalize(t.float().to(device), dim=-1)
        doc_mask[i, :L]   = True
    return doc_matrix, doc_mask


doc_matrix, doc_mask = build_global_doc_matrix(global_items, DEVICE)
torch.cuda.synchronize(DEVICE)
print(f">>> doc_matrix: {tuple(doc_matrix.shape)}  dtype={doc_matrix.dtype}")
print(f">>> GPU mem allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ----------------------------------------------------------------------------
# LOAD QUERIES + QRELS, MAP GT → GLOBAL INDICES
# Score ∈ {1, 2} đều tính là relevant.
# Query không có GT trong global corpus (vd GT page bị skip ở Pipeline A) → drop.
# ----------------------------------------------------------------------------
def load_queries_and_qrels(domains, domain_to_global):
    all_rows   = []
    n_dropped  = 0
    n_total    = 0
    for d in domains:
        dname = d['name']
        q_df  = pd.concat([pd.read_parquet(f) for f in d['queries_files']],
                          ignore_index=True)
        qr_df = pd.concat([pd.read_parquet(f) for f in d['qrels_files']],
                          ignore_index=True)
        qr_df = qr_df[qr_df['score'].isin([1, 2])]

        # Map local query_id → set(local corpus_id)
        local_gt_map = qr_df.groupby('query_id')['corpus_id'].apply(set).to_dict()

        for _, row in q_df.iterrows():
            n_total += 1
            qid = int(row['query_id'])
            local_gt = local_gt_map.get(qid, set())
            gt_global = {
                domain_to_global[(dname, int(lcid))]
                for lcid in local_gt
                if (dname, int(lcid)) in domain_to_global
            }
            if not gt_global:
                n_dropped += 1
                continue
            all_rows.append({
                'domain':        dname,
                'query_id':      qid,
                'query':         row['query'],
                'language':      row['language'],
                'gt_global_idx': gt_global,
            })
    df = pd.DataFrame(all_rows).reset_index(drop=True)
    df['row_idx'] = df.index  # stable key cho query_cache
    return df, n_total, n_dropped


print("\n>>> Loading queries + qrels...")
queries_df, n_total_q, n_dropped = load_queries_and_qrels(domains, domain_to_global)
print(f">>> Total queries loaded          : {n_total_q}")
print(f">>> Dropped (no GT in global idx) : {n_dropped}")
print(f">>> Queries available for eval    : {len(queries_df)}")
print(f"\n>>> Per domain:")
print(queries_df['domain'].value_counts().to_string())
print(f"\n>>> Per language:")
print(queries_df['language'].value_counts().to_string())

# ----------------------------------------------------------------------------
# METRICS HELPERS
# ----------------------------------------------------------------------------
def first_hit(top_k, gt):
    for r, i in enumerate(top_k):
        if i in gt:
            return r + 1
    return -1

def compute_ndcg(ranked, gt, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in gt)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), k)))
    return dcg / idcg if idcg > 0 else 0.0

def hit_metrics(top10, gt):
    h = first_hit(top10, gt)
    return {
        'r1':  int(h != -1 and h <= 1),
        'r5':  int(h != -1 and h <= 5),
        'r10': int(h != -1 and h <= 10),
        'n1':  float(compute_ndcg(top10, gt, 1)),
        'n5':  float(compute_ndcg(top10, gt, 5)),
        'n10': float(compute_ndcg(top10, gt, 10)),
    }

def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0,
            'n1': 0.0, 'n5': 0.0, 'n10': 0.0, 'count': 0}

def _add_metric(dst, src):
    for k in ('r1', 'r5', 'r10'):
        dst[k] += int(src[k])
    for k in ('n1', 'n5', 'n10'):
        dst[k] += float(src[k])
    dst['count'] += 1

def _aggregate(metric_dict):
    cnt = metric_dict['count'] or 1
    return {
        'r@1':     round(metric_dict['r1']  / cnt * 100, 4),
        'r@5':     round(metric_dict['r5']  / cnt * 100, 4),
        'r@10':    round(metric_dict['r10'] / cnt * 100, 4),
        'ndcg@1':  round(metric_dict['n1']  / cnt,       6),
        'ndcg@5':  round(metric_dict['n5']  / cnt,       6),
        'ndcg@10': round(metric_dict['n10'] / cnt,       6),
        'count':   cnt,
    }

# ----------------------------------------------------------------------------
# CORE MAXSIM (float32 trên GPU)
# ----------------------------------------------------------------------------
@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """q_norm: (Q, D). Returns (Q, N_docs)."""
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values

# ----------------------------------------------------------------------------
# CONTENT MASK — loại BOS/EOS/PAD/special tokens
# ----------------------------------------------------------------------------
def build_content_mask(inputs, processor):
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()
    tok = getattr(processor, 'tokenizer', processor)
    special_ids = set()
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()
    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()

# ----------------------------------------------------------------------------
# LATENCY TIMER — GPU event based, research grade
# Dùng:
#     t = LatencyTimer()
#     with t:
#         ... GPU ops ...
#     ms = t.elapsed_ms
# ----------------------------------------------------------------------------
class LatencyTimer:
    def __init__(self, device=DEVICE):
        self.device    = device
        self.start_evt = torch.cuda.Event(enable_timing=True)
        self.end_evt   = torch.cuda.Event(enable_timing=True)
        self.elapsed_ms = None
    def __enter__(self):
        torch.cuda.synchronize(self.device)
        self.start_evt.record()
        return self
    def __exit__(self, *args):
        self.end_evt.record()
        torch.cuda.synchronize(self.device)
        self.elapsed_ms = self.start_evt.elapsed_time(self.end_evt)

# ----------------------------------------------------------------------------
# SAVE HELPER — 3 CSVs per method
# ----------------------------------------------------------------------------
def save_method_results(method_name, query_rows, all_metric, domain_metric,
                         latencies_ms, output_dir=OUTPUT_DIR):
    """
    Lưu 3 CSV:
      - {method}_queries.csv      : per-query results
      - {method}_summary.csv      : aggregate metrics (1 row)
      - {method}_per_domain.csv   : aggregate per domain
    """
    # per-query
    pd.DataFrame(query_rows).to_csv(
        os.path.join(output_dir, f"{method_name}_queries.csv"), index=False)

    # summary
    agg = _aggregate(all_metric)
    agg['method']      = method_name
    agg['latency_median_ms'] = float(np.median(latencies_ms)) if latencies_ms else None
    agg['latency_p95_ms']    = float(np.percentile(latencies_ms, 95)) if latencies_ms else None
    agg['latency_mean_ms']   = float(np.mean(latencies_ms)) if latencies_ms else None
    pd.DataFrame([agg]).to_csv(
        os.path.join(output_dir, f"{method_name}_summary.csv"), index=False)

    # per-domain
    rows = []
    for dname in sorted(domain_metric.keys()):
        r = _aggregate(domain_metric[dname])
        r['domain'] = dname
        r['method'] = method_name
        rows.append(r)
    pd.DataFrame(rows).to_csv(
        os.path.join(output_dir, f"{method_name}_per_domain.csv"), index=False)

print("\n>>> CELL 1 — Shared utilities READY")

In [ ]:
# ============================================================================
# CELL 2 — ENCODE ALL QUERIES ONCE (SDPA mode)
# Cache: proj [S, 128] + content_mask + input_ids → reused by 5 non-OURS methods
# OURS dùng riêng forward eager nên không dùng cache nà0y.
# ============================================================================
print(">>> CELL 2 — Caching query embeddings")

QUERY_CACHE_PATH = os.path.join(OUTPUT_DIR, "_query_cache.pkl")


def encode_all_queries(model, processor, queries_df, device, save_path=None):
    """
    Encode mỗi query 1 lần, cache:
        proj         : Tensor[S, 128] float32 (CPU)
        content_mask : Tensor[S]      float32 (CPU)
        input_ids    : Tensor[S]      int64   (CPU)
    Key cache = row_idx của queries_df (đã reset_index stable).
    """
    if save_path and os.path.exists(save_path):
        print(f"  ✅ Loading cached: {save_path}")
        with open(save_path, 'rb') as f:
            return pickle.load(f)

    model.eval()
    cache    = {}
    n_errors = 0

    for row_idx, row in tqdm(queries_df.iterrows(),
                              total=len(queries_df),
                              desc="Encoding queries"):
        question = row['query']
        try:
            q_inputs = processor.process_texts([question]).to(device)
            with torch.inference_mode():
                out = model(**q_inputs)   # Tensor[1, S, 128] bf16
            proj            = out[0]
            content_mask_1d = build_content_mask(q_inputs, processor)[0]
            input_ids       = q_inputs['input_ids'][0]

            cache[int(row_idx)] = {
                'proj':         proj.detach().cpu().to(torch.float32),
                'content_mask': content_mask_1d.detach().cpu().float(),
                'input_ids':    input_ids.detach().cpu(),
            }
        except Exception as e:
            n_errors += 1
            if n_errors <= 5:
                print(f"  ⚠️  Row {row_idx} failed: {e}")
            continue

    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump(cache, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  💾 Saved: {save_path}")
    if n_errors:
        print(f"  ⚠️  Total errors: {n_errors}")
    return cache


query_cache = encode_all_queries(model, processor, queries_df, DEVICE,
                                   save_path=QUERY_CACHE_PATH)

# Stats
seq_lens = [v['proj'].shape[0] for v in query_cache.values()]
print(f"\n>>> Cached {len(query_cache)} / {len(queries_df)} queries")
print(f">>> Seq length: mean={np.mean(seq_lens):.1f}  "
      f"median={int(np.median(seq_lens))}  max={max(seq_lens)}  min={min(seq_lens)}")
print(f">>> Cache memory (CPU): "
      f"~{sum(v['proj'].numel()*4 for v in query_cache.values())/1e6:.1f} MB")
print(">>> CELL 2 DONE")

In [ ]:
# ============================================================================
# CELL 3 — METHOD 1: FULL (baseline — không pruning/pooling)
# Dùng tất cả content tokens của query, MaxSim.sum() truyền thống.
# ============================================================================
print(">>> CELL 3 — Method 1: FULL baseline")

METHOD_NAME = "full"

# Counters
all_metric     = _init_metric()
domain_metric  = {}
query_rows     = []
latencies_ms   = []

# ----------------------------------------------------------------------------
# Warmup — 50 queries đầu không log, chỉ để CUDA kernel stabilize
# ----------------------------------------------------------------------------
N_WARMUP = 50
N_docs   = doc_matrix.shape[0]

def run_full_query(cached, gt, measure_latency=False):
    """
    Một lượt full-baseline cho 1 query.
    Returns: metric_dict, latency_ms (or None nếu không đo)
    """
    proj         = cached['proj'].to(DEVICE)            # [S, 128]
    content_mask = cached['content_mask'].to(DEVICE)    # [S]

    valid_idx = torch.where(content_mask > 0)[0]
    if valid_idx.numel() == 0:
        # fallback: dùng toàn bộ proj
        q_norm = F.normalize(proj, dim=-1)
    else:
        q_norm = F.normalize(proj[valid_idx], dim=-1)

    if measure_latency:
        with LatencyTimer() as t:
            M      = fast_maxsim(q_norm, doc_matrix, doc_mask)   # [Sq, N]
            scores = M.sum(dim=0)                                # [N]
            top10  = torch.topk(scores, min(10, N_docs)).indices
            top10  = top10.cpu().tolist()
        ms = t.elapsed_ms
    else:
        M      = fast_maxsim(q_norm, doc_matrix, doc_mask)
        scores = M.sum(dim=0)
        top10  = torch.topk(scores, min(10, N_docs)).indices.cpu().tolist()
        ms = None

    return hit_metrics(top10, gt), ms


# ----------------------------------------------------------------------------
# Warmup pass
# ----------------------------------------------------------------------------
print(f"  Warmup {N_WARMUP} queries (not measured)...")
warmup_ids = queries_df.index[:N_WARMUP].tolist()
for ri in tqdm(warmup_ids, desc="Warmup", leave=False):
    if ri not in query_cache:
        continue
    gt = queries_df.at[ri, 'gt_global_idx']
    _ = run_full_query(query_cache[ri], gt, measure_latency=False)

# ----------------------------------------------------------------------------
# Main eval pass — đo latency + metrics
# ----------------------------------------------------------------------------
print(f"  Main eval on {len(queries_df)} queries...")
for ri in tqdm(queries_df.index, desc=f"{METHOD_NAME}", leave=True):
    if ri not in query_cache:
        continue

    row  = queries_df.loc[ri]
    gt   = row['gt_global_idx']
    dom  = row['domain']

    m, ms = run_full_query(query_cache[ri], gt, measure_latency=True)

    # Aggregate
    _add_metric(all_metric, m)
    if dom not in domain_metric:
        domain_metric[dom] = _init_metric()
    _add_metric(domain_metric[dom], m)
    latencies_ms.append(ms)

    query_rows.append({
        'method':    METHOD_NAME,
        'row_idx':   int(ri),
        'domain':    dom,
        'language':  row['language'],
        'query_id':  int(row['query_id']),
        'r@1':       m['r1'],
        'r@5':       m['r5'],
        'r@10':      m['r10'],
        'ndcg@1':    round(m['n1'],  4),
        'ndcg@5':    round(m['n5'],  4),
        'ndcg@10':   round(m['n10'], 4),
        'latency_ms': round(ms, 3),
    })

# ----------------------------------------------------------------------------
# Save
# ----------------------------------------------------------------------------
save_method_results(METHOD_NAME, query_rows, all_metric, domain_metric, latencies_ms)

# Print summary
agg = _aggregate(all_metric)
print(f"\n>>> {METHOD_NAME.upper()} RESULTS")
print(f"  count        : {agg['count']}")
print(f"  R@1          : {agg['r@1']:.2f}%")
print(f"  R@5          : {agg['r@5']:.2f}%")
print(f"  R@10         : {agg['r@10']:.2f}%")
print(f"  nDCG@5       : {agg['ndcg@5']:.4f}")
print(f"  nDCG@10      : {agg['ndcg@10']:.4f}")
print(f"  latency mean : {np.mean(latencies_ms):.2f} ms/q")
print(f"  latency med  : {np.median(latencies_ms):.2f} ms/q")
print(f"  latency p95  : {np.percentile(latencies_ms, 95):.2f} ms/q")
print(">>> CELL 3 DONE")

In [ ]:
# ============================================================================
# CELL 4a — BUILD ATTENTION CACHE (ColPali / PaliGemma)
# PaliGemma structure:
#   model.model.language_model.model.layers → 18 GemmaDecoderLayers
# Gọi parent class forward với output_attentions=True để lấy attention tensors.
# ============================================================================
print(">>> CELL 4a — Building attention cache (ColPali)")

N_LAYERS_CACHE = 9   # ColPali có 18 layers, OURS dùng 9 (≈50%)

# Free embeddings from global_items
try:
    for item in global_items:
        item.pop('embedding', None)
    gc.collect()
except Exception:
    pass


def _get_parent_forward(model):
    """
    Tìm parent class có forward() trả về attentions.
    ColPali.forward() chỉ return proj, nên phải gọi parent.
    """
    for cls in type(model).__mro__[1:]:
        if hasattr(cls, 'forward') and cls.__name__ != 'Module':
            return cls
    raise RuntimeError("Cannot find parent class with forward()")


def build_attention_cache_colpali(model, processor, queries_df, device,
                                   n_last_layers=N_LAYERS_CACHE):
    cache = {}
    n_errors = 0

    # ColPali wraps PaliGemmaForConditionalGeneration as model.model
    inner_model = model.model  # đây là PaliGemmaForConditionalGeneration
    print(f"  Using inner model: {type(inner_model).__name__}")

    inner_model.eval()
    for row_idx, row in tqdm(queries_df.iterrows(),
                              total=len(queries_df),
                              desc="Encoding + attention"):
        question = row['query']
        try:
            q_inputs = processor.process_texts([question]).to(device)

            with torch.inference_mode():
                outputs = inner_model(
                    input_ids=q_inputs['input_ids'],
                    attention_mask=q_inputs['attention_mask'],
                    use_cache=False,
                    output_attentions=True,
                    output_hidden_states=False,
                    return_dict=True,
                )
                attentions = outputs.attentions
                if attentions is None or len(attentions) == 0:
                    raise ValueError("attentions is None — eager mode may not be active")

                last_attns = attentions[-n_last_layers:]
                cache[int(row_idx)] = [
                    a.detach().to('cpu', dtype=torch.bfloat16).clone()
                    for a in last_attns
                ]
        except Exception as e:
            n_errors += 1
            if n_errors <= 5:
                print(f"  ⚠️  Row {row_idx}: {e}")
            if n_errors == 5:
                print(f"  (suppressing further error messages...)")
            continue

    if n_errors:
        print(f"  ⚠️  Total errors: {n_errors}")
    return cache


attention_cache = build_attention_cache_colpali(
    model, processor, queries_df, DEVICE, n_last_layers=N_LAYERS_CACHE
)

if attention_cache:
    sample = next(iter(attention_cache.values()))
    print(f"\n>>> Cached {len(attention_cache)} queries × {len(sample)} layers")
    print(f">>> Per-layer shape: {tuple(sample[0].shape)}  dtype={sample[0].dtype}")
    total_mb = sum(sum(t.numel() * 2 for t in v) for v in attention_cache.values()) / 1e6
    print(f">>> Attention cache RAM: ~{total_mb:.1f} MB")
else:
    print(">>> ⚠️ Attention cache EMPTY — Cell 4 và Cell 8 sẽ skip queries")

print(">>> CELL 4a DONE")

In [ ]:
# ============================================================================
# CELL 4 — METHOD 2: ATTENTION SCORE PRUNING (Lassance et al. 2021)
# i(t) = Σ_h Σ_j A[h, j, t]  — column sum over last layer attention
# Paper default: n_last_layers = 1
# Keep top TOPK_RATIO * N tokens, discard rest, sum MaxSim on kept.
# ============================================================================
print(">>> CELL 4 — Method 2: Attention Score Pruning")

METHOD_NAME     = "attention_pruning"
N_LAST_ATTN     = 1   # paper-exact (last layer only)

all_metric    = _init_metric()
domain_metric = {}
query_rows    = []
latencies_ms  = []
N_docs        = doc_matrix.shape[0]


def run_attention_pruning(cached, attn_cached, n_last_layers=N_LAST_ATTN):
    with LatencyTimer() as t:
        proj         = cached['proj'].to(DEVICE)
        content_mask = cached['content_mask'].to(DEVICE)
        # Lấy last_n attention layers, move lên GPU
        attns = [a.to(DEVICE, dtype=torch.float32)
                 for a in attn_cached[-n_last_layers:]]

        valid_idx = torch.where(content_mask > 0)[0]
        if valid_idx.numel() == 0:
            valid_idx = torch.arange(proj.shape[0], device=DEVICE)

        q_norm = F.normalize(proj[valid_idx].float(), dim=-1)
        N_tok  = q_norm.shape[0]
        n_keep = max(1, round(N_tok * TOPK_RATIO))

        # Importance = Σ_h Σ_j A[h, j, t] = column sum over (H, sources)
        # shape a: [1, H, S, S] — index [1][h][j=source][t=target]
        imp_full = torch.zeros(content_mask.shape[0], device=DEVICE,
                                dtype=torch.float32)
        for a in attns:
            imp_full = imp_full + a[0].sum(dim=(0, 1))   # sum over H + sources
        imp_full = imp_full / len(attns)
        imp_full = imp_full * content_mask               # zero-out specials
        imp_valid = imp_full[valid_idx]

        # Top-k kept
        topk_idx = torch.topk(imp_valid, n_keep).indices
        q_kept   = q_norm[topk_idx]

        # MaxSim sum
        M      = fast_maxsim(q_kept, doc_matrix, doc_mask)
        scores = M.sum(dim=0)
        top10  = torch.topk(scores, min(10, N_docs)).indices.cpu().tolist()

    return top10, t.elapsed_ms


# Warmup
print(f"  Warmup 50 queries (not measured)...")
warmup_ids = queries_df.index[:50].tolist()
for ri in tqdm(warmup_ids, desc="Warmup", leave=False):
    if ri not in query_cache or ri not in attention_cache:
        continue
    _ = run_attention_pruning(query_cache[ri], attention_cache[ri])

# Main eval
print(f"  Main eval on {len(queries_df)} queries...")
for ri in tqdm(queries_df.index, desc=METHOD_NAME):
    if ri not in query_cache or ri not in attention_cache:
        continue
    row = queries_df.loc[ri]
    gt  = row['gt_global_idx']
    dom = row['domain']

    top10, ms = run_attention_pruning(query_cache[ri], attention_cache[ri])
    m = hit_metrics(top10, gt)

    _add_metric(all_metric, m)
    if dom not in domain_metric:
        domain_metric[dom] = _init_metric()
    _add_metric(domain_metric[dom], m)
    latencies_ms.append(ms)

    query_rows.append({
        'method':   METHOD_NAME, 'row_idx': int(ri),
        'domain':   dom, 'language': row['language'],
        'query_id': int(row['query_id']),
        'r@1':      m['r1'],  'r@5':  m['r5'],  'r@10':  m['r10'],
        'ndcg@1':   round(m['n1'], 4),
        'ndcg@5':   round(m['n5'], 4),
        'ndcg@10':  round(m['n10'], 4),
        'latency_ms': round(ms, 3),
    })

save_method_results(METHOD_NAME, query_rows, all_metric, domain_metric, latencies_ms)

agg = _aggregate(all_metric)
print(f"\n>>> {METHOD_NAME.upper()} RESULTS (n_last={N_LAST_ATTN}, ratio={TOPK_RATIO})")
print(f"  count        : {agg['count']}")
print(f"  R@1/5/10     : {agg['r@1']:.2f}% / {agg['r@5']:.2f}% / {agg['r@10']:.2f}%")
print(f"  nDCG@5/10    : {agg['ndcg@5']:.4f} / {agg['ndcg@10']:.4f}")
print(f"  latency med  : {np.median(latencies_ms):.2f} ms/q")
print(f"  latency p95  : {np.percentile(latencies_ms, 95):.2f} ms/q")
print(">>> CELL 4 DONE")

In [ ]:
# ============================================================================
# CELL 5 — METHOD 3: RANDOM PRUNING (baseline)
# Chọn ngẫu nhiên TOPK_RATIO * N tokens, MaxSim sum.
# Fix seed cho reproducibility.
# ============================================================================
print(">>> CELL 5 — Method 3: Random Pruning")

METHOD_NAME = "random_pruning"
RANDOM_SEED = 42

all_metric    = _init_metric()
domain_metric = {}
query_rows    = []
latencies_ms  = []

# Fixed generator cho reproducibility
rng = torch.Generator(device=DEVICE)
rng.manual_seed(RANDOM_SEED)


def run_random_pruning(cached):
    with LatencyTimer() as t:
        proj         = cached['proj'].to(DEVICE)
        content_mask = cached['content_mask'].to(DEVICE)

        valid_idx = torch.where(content_mask > 0)[0]
        if valid_idx.numel() == 0:
            valid_idx = torch.arange(proj.shape[0], device=DEVICE)

        q_norm = F.normalize(proj[valid_idx].float(), dim=-1)
        N_tok  = q_norm.shape[0]
        n_keep = max(1, round(N_tok * TOPK_RATIO))

        perm    = torch.randperm(N_tok, device=DEVICE, generator=rng)
        kept_ix = perm[:n_keep]
        q_kept  = q_norm[kept_ix]

        M      = fast_maxsim(q_kept, doc_matrix, doc_mask)
        scores = M.sum(dim=0)
        top10  = torch.topk(scores, min(10, N_docs)).indices.cpu().tolist()

    return top10, t.elapsed_ms


print(f"  Warmup 50 queries...")
for ri in tqdm(queries_df.index[:50].tolist(), desc="Warmup", leave=False):
    if ri not in query_cache:
        continue
    _ = run_random_pruning(query_cache[ri])

print(f"  Main eval on {len(queries_df)} queries...")
for ri in tqdm(queries_df.index, desc=METHOD_NAME):
    if ri not in query_cache:
        continue
    row = queries_df.loc[ri]
    gt  = row['gt_global_idx']
    dom = row['domain']

    top10, ms = run_random_pruning(query_cache[ri])
    m = hit_metrics(top10, gt)

    _add_metric(all_metric, m)
    if dom not in domain_metric:
        domain_metric[dom] = _init_metric()
    _add_metric(domain_metric[dom], m)
    latencies_ms.append(ms)

    query_rows.append({
        'method':   METHOD_NAME, 'row_idx': int(ri),
        'domain':   dom, 'language': row['language'],
        'query_id': int(row['query_id']),
        'r@1':      m['r1'], 'r@5':   m['r5'],  'r@10':  m['r10'],
        'ndcg@1':   round(m['n1'], 4),
        'ndcg@5':   round(m['n5'], 4),
        'ndcg@10':  round(m['n10'], 4),
        'latency_ms': round(ms, 3),
    })

save_method_results(METHOD_NAME, query_rows, all_metric, domain_metric, latencies_ms)

agg = _aggregate(all_metric)
print(f"\n>>> {METHOD_NAME.upper()} RESULTS (ratio={TOPK_RATIO}, seed={RANDOM_SEED})")
print(f"  count        : {agg['count']}")
print(f"  R@1/5/10     : {agg['r@1']:.2f}% / {agg['r@5']:.2f}% / {agg['r@10']:.2f}%")
print(f"  nDCG@5/10    : {agg['ndcg@5']:.4f} / {agg['ndcg@10']:.4f}")
print(f"  latency med  : {np.median(latencies_ms):.2f} ms/q")
print(f"  latency p95  : {np.percentile(latencies_ms, 95):.2f} ms/q")
print(">>> CELL 5 DONE")

In [ ]:
# ============================================================================
# CELL 6 — METHOD 4: HIERARCHICAL POOLING (Ward's agglomerative)
# K = round(N * TOPK_RATIO) cụm, mean-pool từng cụm → K vectors, MaxSim sum.
# ============================================================================
print(">>> CELL 6 — Method 4: Hierarchical Pooling (Ward)")

METHOD_NAME = "hierarchical_pooling"

all_metric    = _init_metric()
domain_metric = {}
query_rows    = []
latencies_ms  = []


def _ward_distance_matrix(cents, sizes):
    sim     = torch.matmul(cents, cents.t()).clamp(-1.0, 1.0)
    sq_dist = 2.0 * (1.0 - sim)
    ni = sizes.float().unsqueeze(1)
    nj = sizes.float().unsqueeze(0)
    w  = (ni * nj) / (ni + nj)
    ward = w * sq_dist
    mask = torch.ones(cents.shape[0], cents.shape[0],
                       dtype=torch.bool, device=cents.device).tril()
    ward.masked_fill_(mask, float('inf'))
    return ward


def ward_pool(vecs, n_clusters):
    N = vecs.shape[0]
    if N <= n_clusters:
        return vecs.clone()
    device = vecs.device
    sums   = vecs.clone().float()
    sizes  = torch.ones(N, device=device, dtype=torch.float32)
    cents  = F.normalize(sums, dim=-1)
    active = torch.ones(N, dtype=torch.bool, device=device)
    C      = N
    while C > n_clusters:
        active_idx = active.nonzero(as_tuple=True)[0]
        c_act = cents[active_idx]
        s_act = sizes[active_idx]
        ward  = _ward_distance_matrix(c_act, s_act)
        flat_idx = ward.argmin().item()
        n_act    = active_idx.shape[0]
        ai = flat_idx // n_act
        aj = flat_idx %  n_act
        gi = active_idx[ai].item()
        gj = active_idx[aj].item()
        sums[gi]   = sums[gi] + sums[gj]
        sizes[gi]  = sizes[gi] + sizes[gj]
        cents[gi]  = F.normalize(sums[gi].unsqueeze(0), dim=-1).squeeze(0)
        active[gj] = False
        C -= 1
    final_idx = active.nonzero(as_tuple=True)[0]
    return cents[final_idx]


def run_ward(cached):
    with LatencyTimer() as t:
        proj         = cached['proj'].to(DEVICE)
        content_mask = cached['content_mask'].to(DEVICE)

        valid_idx = torch.where(content_mask > 0)[0]
        if valid_idx.numel() == 0:
            valid_idx = torch.arange(proj.shape[0], device=DEVICE)

        q_norm = F.normalize(proj[valid_idx].float(), dim=-1)
        N_tok  = q_norm.shape[0]
        K      = max(1, round(N_tok * TOPK_RATIO))

        centroids = ward_pool(q_norm, K)

        M      = fast_maxsim(centroids, doc_matrix, doc_mask)
        scores = M.sum(dim=0)
        top10  = torch.topk(scores, min(10, N_docs)).indices.cpu().tolist()

    return top10, t.elapsed_ms


print(f"  Warmup 50 queries...")
for ri in tqdm(queries_df.index[:50].tolist(), desc="Warmup", leave=False):
    if ri not in query_cache:
        continue
    _ = run_ward(query_cache[ri])

print(f"  Main eval on {len(queries_df)} queries...")
for ri in tqdm(queries_df.index, desc=METHOD_NAME):
    if ri not in query_cache:
        continue
    row = queries_df.loc[ri]
    gt  = row['gt_global_idx']
    dom = row['domain']

    top10, ms = run_ward(query_cache[ri])
    m = hit_metrics(top10, gt)

    _add_metric(all_metric, m)
    if dom not in domain_metric:
        domain_metric[dom] = _init_metric()
    _add_metric(domain_metric[dom], m)
    latencies_ms.append(ms)

    query_rows.append({
        'method':   METHOD_NAME, 'row_idx': int(ri),
        'domain':   dom, 'language': row['language'],
        'query_id': int(row['query_id']),
        'r@1':      m['r1'], 'r@5':   m['r5'],  'r@10':  m['r10'],
        'ndcg@1':   round(m['n1'], 4),
        'ndcg@5':   round(m['n5'], 4),
        'ndcg@10':  round(m['n10'], 4),
        'latency_ms': round(ms, 3),
    })

save_method_results(METHOD_NAME, query_rows, all_metric, domain_metric, latencies_ms)

agg = _aggregate(all_metric)
print(f"\n>>> {METHOD_NAME.upper()} RESULTS (K = round(N * {TOPK_RATIO}))")
print(f"  count        : {agg['count']}")
print(f"  R@1/5/10     : {agg['r@1']:.2f}% / {agg['r@5']:.2f}% / {agg['r@10']:.2f}%")
print(f"  nDCG@5/10    : {agg['ndcg@5']:.4f} / {agg['ndcg@10']:.4f}")
print(f"  latency med  : {np.median(latencies_ms):.2f} ms/q")
print(f"  latency p95  : {np.percentile(latencies_ms, 95):.2f} ms/q")
print(">>> CELL 6 DONE")

In [ ]:
# ============================================================================
# CELL 7 — METHOD 5: SPHERICAL KMEANS POOLING
# K = round(N * TOPK_RATIO) clusters, 10 iterations, fixed seed.
# ============================================================================
print(">>> CELL 7 — Method 5: Spherical KMeans")

METHOD_NAME  = "kmeans_pooling"
KMEANS_ITERS = 10
KMEANS_SEED  = 42

all_metric    = _init_metric()
domain_metric = {}
query_rows    = []
latencies_ms  = []

_kmeans_rng = torch.Generator(device=DEVICE)
_kmeans_rng.manual_seed(KMEANS_SEED)


def spherical_kmeans(X, K, n_iters=KMEANS_ITERS):
    N, D = X.shape
    K = min(K, N)
    if K >= N:
        return X.clone()

    perm      = torch.randperm(N, device=X.device, generator=_kmeans_rng)
    centroids = X[perm[:K]].clone()

    for _ in range(n_iters):
        sim         = torch.mm(X, centroids.t())      # [N, K]
        cluster_ids = sim.argmax(dim=1)               # [N]

        # Vectorized update: scatter-sum
        new_centroids = torch.zeros_like(centroids)
        counts = torch.zeros(K, device=X.device, dtype=torch.float32)
        new_centroids.index_add_(0, cluster_ids, X)
        counts.index_add_(0, cluster_ids,
                           torch.ones(N, device=X.device, dtype=torch.float32))
        non_empty = counts > 0
        new_centroids[non_empty] = new_centroids[non_empty] / counts[non_empty].unsqueeze(1)
        new_centroids[~non_empty] = centroids[~non_empty]
        centroids = F.normalize(new_centroids, dim=-1)
    return centroids


def run_kmeans(cached):
    with LatencyTimer() as t:
        proj         = cached['proj'].to(DEVICE)
        content_mask = cached['content_mask'].to(DEVICE)

        valid_idx = torch.where(content_mask > 0)[0]
        if valid_idx.numel() == 0:
            valid_idx = torch.arange(proj.shape[0], device=DEVICE)

        q_norm = F.normalize(proj[valid_idx].float(), dim=-1)
        N_tok  = q_norm.shape[0]
        K      = max(1, round(N_tok * TOPK_RATIO))

        centroids = spherical_kmeans(q_norm, K)

        M      = fast_maxsim(centroids, doc_matrix, doc_mask)
        scores = M.sum(dim=0)
        top10  = torch.topk(scores, min(10, N_docs)).indices.cpu().tolist()

    return top10, t.elapsed_ms


print(f"  Warmup 50 queries...")
for ri in tqdm(queries_df.index[:50].tolist(), desc="Warmup", leave=False):
    if ri not in query_cache:
        continue
    _ = run_kmeans(query_cache[ri])

print(f"  Main eval on {len(queries_df)} queries...")
for ri in tqdm(queries_df.index, desc=METHOD_NAME):
    if ri not in query_cache:
        continue
    row = queries_df.loc[ri]
    gt  = row['gt_global_idx']
    dom = row['domain']

    top10, ms = run_kmeans(query_cache[ri])
    m = hit_metrics(top10, gt)

    _add_metric(all_metric, m)
    if dom not in domain_metric:
        domain_metric[dom] = _init_metric()
    _add_metric(domain_metric[dom], m)
    latencies_ms.append(ms)

    query_rows.append({
        'method':   METHOD_NAME, 'row_idx': int(ri),
        'domain':   dom, 'language': row['language'],
        'query_id': int(row['query_id']),
        'r@1':      m['r1'], 'r@5':   m['r5'],  'r@10':  m['r10'],
        'ndcg@1':   round(m['n1'], 4),
        'ndcg@5':   round(m['n5'], 4),
        'ndcg@10':  round(m['n10'], 4),
        'latency_ms': round(ms, 3),
    })

save_method_results(METHOD_NAME, query_rows, all_metric, domain_metric, latencies_ms)

agg = _aggregate(all_metric)
print(f"\n>>> {METHOD_NAME.upper()} RESULTS (K={TOPK_RATIO}, iters={KMEANS_ITERS})")
print(f"  count        : {agg['count']}")
print(f"  R@1/5/10     : {agg['r@1']:.2f}% / {agg['r@5']:.2f}% / {agg['r@10']:.2f}%")
print(f"  nDCG@5/10    : {agg['ndcg@5']:.4f} / {agg['ndcg@10']:.4f}")
print(f"  latency med  : {np.median(latencies_ms):.2f} ms/q")
print(f"  latency p95  : {np.percentile(latencies_ms, 95):.2f} ms/q")
print(">>> CELL 7 DONE")

In [ ]:
# ============================================================================
# CELL 8 — METHOD 6: OURS (SVD sink importance + cluster pool, minimal)
# Config cố định (1 config duy nhất cho bảng final):
#   n_last_layers = 16
#   normalize     = pre  (normalize pool vector TRƯỚC MaxSim)
#   scoring       = weighted (importance-weighted sum)
#   TOPK_RATIO    = 0.5
# ============================================================================
print(">>> CELL 8 — Method 6: OURS (minimal)")

METHOD_NAME       = "ours"
OURS_N_LAST       = 9
OURS_TEMPERATURE  = 0.5
OURS_SVD_RANK     = 1

all_metric    = _init_metric()
domain_metric = {}
query_rows    = []
latencies_ms  = []


def remove_sink_components_batch(attn_heads, k=1):
    """attn_heads: [H, S, S] float → trả về cleaned [H, S, S]."""
    try:
        U, S, Vh = torch.linalg.svd(attn_heads, full_matrices=False)
        k = min(k, S.shape[-1])
        sink = (U[..., :k] * S[:, :k].unsqueeze(1)) @ Vh[:, :k, :]
        return (attn_heads - sink).clamp(min=0.0)
    except Exception:
        return attn_heads


def compute_svd_importance(attentions, content_mask, layer_weights,
                             temperature=OURS_TEMPERATURE, k=OURS_SVD_RANK):
    """
    attentions   : list of [1, H, S, S]
    content_mask : [S]
    layer_weights: [len(attentions)]  — đã normalize sum = 1
    Returns      : importance [S]
    """
    S      = content_mask.shape[0]
    device = content_mask.device
    importance = torch.zeros(S, device=device, dtype=torch.float32)

    for i, a in enumerate(attentions):
        a = a.float()                   # [1, H, S, S]
        attn_flat = a[0]                # [H, S, S]
        cleaned   = remove_sink_components_batch(attn_flat, k)   # [H, S, S]
        # Column sum (sources) + mean over heads
        layer_imp = cleaned.sum(dim=1).mean(dim=0)   # [S]
        layer_imp = layer_imp * content_mask
        m = layer_imp.max().clamp(min=1e-8)
        layer_imp = layer_imp / m
        importance = importance + layer_weights[i] * layer_imp

    importance = importance * content_mask
    importance = F.softplus(importance / temperature)
    importance = importance * content_mask
    return importance


def build_cluster_pool_pre_weighted(q_kept, q_disc, imp_kept, imp_disc,
                                      doc_matrix, doc_mask):
    """
    Pre-normalize + importance-weighted cluster pooling.
    Returns: (cluster_scores [N_docs], total_disc_imp: float)
    """
    n_docs = doc_matrix.shape[0]
    if q_disc.shape[0] == 0:
        return torch.zeros(n_docs, device=doc_matrix.device), 0.0

    # Assign mỗi disc token về kept token gần nhất (cosine)
    sim_assign  = torch.mm(q_disc, q_kept.t())
    cluster_ids = sim_assign.argmax(dim=-1)

    cluster_scores = torch.zeros(n_docs, device=doc_matrix.device)
    total_disc_imp = imp_disc.sum().item()

    for c in cluster_ids.unique():
        members = (cluster_ids == c).nonzero(as_tuple=True)[0]
        w       = imp_disc[members]
        w_sum   = w.sum().clamp(min=1e-8)
        w_norm  = w / w_sum
        pool_vec = (q_disc[members] * w_norm.unsqueeze(-1)).sum(dim=0)

        # PRE-normalize
        pool_vec = F.normalize(pool_vec.unsqueeze(0), dim=-1)
        sim = torch.einsum('qd,nld->qnl', pool_vec, doc_matrix)
        sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
        ms = sim.max(dim=-1).values.squeeze(0)
        cluster_scores += ms * w_sum.item()

    if total_disc_imp > 1e-8:
        cluster_scores = cluster_scores / total_disc_imp
    return cluster_scores, total_disc_imp


def run_ours(cached, attn_cached, n_last_layers=OURS_N_LAST):
    with LatencyTimer() as t:
        proj         = cached['proj'].to(DEVICE)
        content_mask = cached['content_mask'].to(DEVICE)
        attns = [a.to(DEVICE, dtype=torch.float32)
                 for a in attn_cached[-n_last_layers:]]
        n_actual = len(attns)

        # Layer weights: exp(linspace(0,1)) normalized → thiên về layer cuối
        layer_weights = torch.exp(
            torch.linspace(0, 1, n_actual, device=DEVICE)
        )
        layer_weights = layer_weights / layer_weights.sum()

        importance_full = compute_svd_importance(
            attns, content_mask, layer_weights
        )

        valid_idx = torch.where(content_mask > 0)[0]
        if valid_idx.numel() == 0:
            valid_idx = torch.arange(proj.shape[0], device=DEVICE)

        q_norm    = F.normalize(proj[valid_idx].float(), dim=-1)
        imp_valid = importance_full[valid_idx]
        N_tok     = q_norm.shape[0]
        n_keep    = max(1, round(N_tok * TOPK_RATIO))

        # Sort by importance desc, keep top n_keep
        sorted_ix = torch.argsort(imp_valid, descending=True)
        kept_ix   = sorted_ix[:n_keep]
        disc_ix   = sorted_ix[n_keep:]

        q_kept   = q_norm[kept_ix]
        q_disc   = q_norm[disc_ix]
        imp_kept = imp_valid[kept_ix]
        imp_disc = imp_valid[disc_ix]

        # MaxSim on kept
        M_kept = fast_maxsim(q_kept, doc_matrix, doc_mask)   # [n_keep, N_docs]

        # Cluster pool for discarded
        cluster_scores, total_disc_imp = build_cluster_pool_pre_weighted(
            q_kept, q_disc, imp_kept, imp_disc, doc_matrix, doc_mask
        )

        # Importance-weighted sum on kept
        imp_kept_n  = imp_kept / imp_kept.sum().clamp(min=1e-8)
        scores_kept = (M_kept * imp_kept_n.unsqueeze(-1)).sum(dim=0)

        pool_frac = total_disc_imp / imp_valid.sum().clamp(min=1e-8).item()
        scores    = scores_kept + cluster_scores * pool_frac

        top10 = torch.topk(scores, min(10, N_docs)).indices.cpu().tolist()

    return top10, t.elapsed_ms


print(f"  Warmup 50 queries...")
for ri in tqdm(queries_df.index[:50].tolist(), desc="Warmup", leave=False):
    if ri not in query_cache or ri not in attention_cache:
        continue
    _ = run_ours(query_cache[ri], attention_cache[ri])

print(f"  Main eval on {len(queries_df)} queries...")
for ri in tqdm(queries_df.index, desc=METHOD_NAME):
    if ri not in query_cache or ri not in attention_cache:
        continue
    row = queries_df.loc[ri]
    gt  = row['gt_global_idx']
    dom = row['domain']

    top10, ms = run_ours(query_cache[ri], attention_cache[ri])
    m = hit_metrics(top10, gt)

    _add_metric(all_metric, m)
    if dom not in domain_metric:
        domain_metric[dom] = _init_metric()
    _add_metric(domain_metric[dom], m)
    latencies_ms.append(ms)

    query_rows.append({
        'method':   METHOD_NAME, 'row_idx': int(ri),
        'domain':   dom, 'language': row['language'],
        'query_id': int(row['query_id']),
        'r@1':      m['r1'], 'r@5':   m['r5'],  'r@10':  m['r10'],
        'ndcg@1':   round(m['n1'], 4),
        'ndcg@5':   round(m['n5'], 4),
        'ndcg@10':  round(m['n10'], 4),
        'latency_ms': round(ms, 3),
    })

save_method_results(METHOD_NAME, query_rows, all_metric, domain_metric, latencies_ms)

agg = _aggregate(all_metric)
print(f"\n>>> {METHOD_NAME.upper()} RESULTS "
      f"(n_last={OURS_N_LAST}, norm=pre, scoring=weighted, ratio={TOPK_RATIO})")
print(f"  count        : {agg['count']}")
print(f"  R@1/5/10     : {agg['r@1']:.2f}% / {agg['r@5']:.2f}% / {agg['r@10']:.2f}%")
print(f"  nDCG@5/10    : {agg['ndcg@5']:.4f} / {agg['ndcg@10']:.4f}")
print(f"  latency med  : {np.median(latencies_ms):.2f} ms/q")
print(f"  latency p95  : {np.percentile(latencies_ms, 95):.2f} ms/q")
print(">>> CELL 8 DONE")

In [ ]:
# ============================================================================
# CELL 9 — MASTER TABLE + VISUALIZATIONS
# Gộp 6 methods, tạo bảng final theo format ảnh của bạn, plot 3 visualization.
# ============================================================================
print(">>> CELL 9 — MASTER consolidation")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Thứ tự hiển thị trong bảng & plot
METHOD_ORDER = [
    ('full',                 'Full'),
    ('attention_pruning',    'Attention score pruning'),
    ('random_pruning',       'Random pruning'),
    ('hierarchical_pooling', 'Hierarchical Pooling'),
    ('kmeans_pooling',       'KMEANS'),
    ('ours',                 'OURS'),
]

# ----------------------------------------------------------------------------
# MASTER summary — giống format ảnh của bạn
# ----------------------------------------------------------------------------
master_rows = []
for method_key, display_name in METHOD_ORDER:
    summary_path = os.path.join(OUTPUT_DIR, f"{method_key}_summary.csv")
    if not os.path.exists(summary_path):
        print(f"  ⚠️  {method_key}_summary.csv không tồn tại — skip")
        continue
    sdf = pd.read_csv(summary_path)
    r = sdf.iloc[0].to_dict()
    master_rows.append({
        'Method':       display_name,
        'R@1':          r['r@1'],
        'R@5':          r['r@5'],
        'R@10':         r['r@10'],
        'nDCG@5':       r['ndcg@5'],
        'nDCG@10':      r['ndcg@10'],
        'Latency(ms/q)': round(r['latency_median_ms'], 2),
    })

master_df = pd.DataFrame(master_rows)
master_path = os.path.join(OUTPUT_DIR, "MASTER_all_methods.csv")
master_df.to_csv(master_path, index=False)

print(f"\n{'='*80}")
print(f"MASTER TABLE — 50% TOKEN (cross-domain, all languages)")
print(f"{'='*80}")
print(master_df.to_string(index=False))
print(f"\n✅ Saved: {master_path}")

# ----------------------------------------------------------------------------
# Merge per-domain results từ tất cả methods
# ----------------------------------------------------------------------------
per_domain_frames = []
for method_key, display_name in METHOD_ORDER:
    pd_path = os.path.join(OUTPUT_DIR, f"{method_key}_per_domain.csv")
    if os.path.exists(pd_path):
        df_ = pd.read_csv(pd_path)
        df_['method_display'] = display_name
        per_domain_frames.append(df_)

if per_domain_frames:
    all_per_domain = pd.concat(per_domain_frames, ignore_index=True)
    all_per_domain.to_csv(
        os.path.join(OUTPUT_DIR, "MASTER_per_domain.csv"), index=False
    )
    print(f"✅ Saved: MASTER_per_domain.csv  ({len(all_per_domain)} rows)")

# ============================================================================
# VISUALIZATIONS
# ============================================================================

# Colors — distinct palette
COLORS = {
    'Full':                    '#444444',
    'Attention score pruning': '#E07B39',
    'Random pruning':          '#888780',
    'Hierarchical Pooling':    '#3B72C4',
    'KMEANS':                  '#9B59B6',
    'OURS':                    '#1DB954',
}

# ----------------------------------------------------------------------------
# Plot 1 — Bar chart: 6 methods × 4 metrics
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
metrics_to_plot = [
    ('R@1',     'R@1',     True),
    ('R@5',     'R@5',     True),
    ('R@10',    'R@10',    True),
    ('nDCG@5',  'nDCG@5',  False),
    ('nDCG@10', 'nDCG@10', False),
]

master_by_key = {row['Method']: row for _, row in master_df.iterrows()}

for ax_idx, (title, col_name, pct) in enumerate(metrics_to_plot):
    ax = axes[ax_idx]
    labels = [d for _, d in METHOD_ORDER if d in master_by_key]
    values = [master_by_key[d][col_name] for d in labels]
    colors = [COLORS[d] for d in labels]

    bars = ax.bar(range(len(labels)), values, color=colors, alpha=0.85,
                   edgecolor='white', linewidth=1.2)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel('%' if pct else 'Score', fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.grid(axis='y', alpha=0.25, linewidth=0.5)
    ax.spines[['top', 'right']].set_visible(False)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f"{v:.2f}" if pct else f"{v:.4f}",
                 ha='center', va='bottom', fontsize=7)

plt.suptitle("ViDoRe V3 × ColPali — 6 Methods @ 50% Token",
              fontsize=13, y=1.03)
plt.tight_layout()
p1 = os.path.join(OUTPUT_DIR, "master_bar_all_methods.png")
plt.savefig(p1, dpi=140, bbox_inches='tight')
plt.close()
print(f"✅ Plot 1 (bar)     → {p1}")

# ----------------------------------------------------------------------------
# Plot 2 — Heatmap: domain × method on nDCG@10
# ----------------------------------------------------------------------------
if per_domain_frames:
    all_per_domain = pd.concat(per_domain_frames, ignore_index=True)
    pivot = all_per_domain.pivot_table(
        index='domain', columns='method_display',
        values='ndcg@10', aggfunc='first'
    )
    # Reorder columns theo METHOD_ORDER
    col_order = [d for _, d in METHOD_ORDER if d in pivot.columns]
    pivot = pivot[col_order]

    fig2, ax2 = plt.subplots(figsize=(
        max(7, len(col_order) * 1.4),
        max(4, len(pivot) * 0.6)
    ))
    im = ax2.imshow(pivot.values, aspect='auto', cmap='YlGn')
    ax2.set_xticks(range(len(col_order)))
    ax2.set_xticklabels(col_order, rotation=30, ha='right', fontsize=9)
    ax2.set_yticks(range(len(pivot)))
    ax2.set_yticklabels(pivot.index, fontsize=9)
    ax2.set_title("nDCG@10 — domain × method", fontsize=11)
    plt.colorbar(im, ax=ax2, label='nDCG@10')
    vmax = pivot.values.max()
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            ax2.text(j, i, f"{v:.3f}", ha='center', va='center',
                      fontsize=7,
                      color='white' if v > vmax * 0.7 else 'black')
    plt.tight_layout()
    p2 = os.path.join(OUTPUT_DIR, "master_heatmap_domain_method.png")
    plt.savefig(p2, dpi=140, bbox_inches='tight')
    plt.close()
    print(f"✅ Plot 2 (heatmap) → {p2}")

# ----------------------------------------------------------------------------
# Plot 3 — Scatter: latency vs nDCG@10 (tradeoff)
# ----------------------------------------------------------------------------
fig3, ax3 = plt.subplots(figsize=(9, 6))
for _, row in master_df.iterrows():
    name = row['Method']
    ax3.scatter(row['Latency(ms/q)'], row['nDCG@10'],
                 s=180, color=COLORS[name], edgecolor='white', linewidth=1.5,
                 label=name, zorder=3)
    ax3.annotate(name,
                  xy=(row['Latency(ms/q)'], row['nDCG@10']),
                  xytext=(6, 6), textcoords='offset points',
                  fontsize=9)

ax3.set_xlabel('Latency (ms/q, median)', fontsize=10)
ax3.set_ylabel('nDCG@10', fontsize=10)
ax3.set_title('Quality vs Latency tradeoff', fontsize=11)
ax3.grid(alpha=0.25, linewidth=0.5)
ax3.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
p3 = os.path.join(OUTPUT_DIR, "master_scatter_latency_quality.png")
plt.savefig(p3, dpi=140, bbox_inches='tight')
plt.close()
print(f"✅ Plot 3 (scatter) → {p3}")

print("\n>>> CELL 9 DONE — TẤT CẢ HOÀN TẤT 🎉")